# AI 서비스 백엔드 개발 (FastAPI) — 종합 과제

---

## 과제 안내

- **배점**: 각 문제 10점, 총 100점 (부분 점수 없음 — PASS / FAIL 방식)
- **제출물**: 아래 두 가지 파일을 함께 제출

| 제출 파일 | 포함 내용 |
|----------|----------|
| `과제.ipynb` | Q1·Q2·Q3·Q6 Colab 실행 결과 포함 / Q4·Q5·Q7·Q8·Q9 캡처 이미지 첨부 / Q10 설계도 이미지 첨부 |
| `project.zip` | `assignment_project/` 전체 디렉토리 (모든 TODO 구현 완료 상태) |

---

## 조건 (미충족 시 해당 문제 0점)

| 문제 |  조건 |
|------|----------|
| Q1 | 동기·비동기 실행 시간 수치 **두 가지 모두** 셀 출력에 포함 |
| Q3 | `latency_ms` 수치가 포함된 ChatResponse JSON 출력 |
| Q4 | `status` / `token` / `source` / `done` 4종 이벤트 수신 터미널 **캡처 이미지** |
| Q5 | 허용 도메인 200 · 차단 도메인 CORS 오류 브라우저 Network 탭 **캡처 이미지 2장** |
| Q7 | `429 Too Many Requests` 응답 **캡처 이미지** (제한 횟수 명시 필수) |
| Q8 | `docker compose ps` 두 서비스 `Up` 상태 **캡처 이미지** |
| Q9 | 검색 결과 문서명 + LLM 답변 + latency 수치 포함 **캡처 이미지** |
| Q10 | draw.io 파일 또는 손그림 사진 **직접 제작** |

---

## 기술 스택 (전 문제 공통)

- **LLM**: `litellm`을 통해 `gemini/gemini-3.1-flash-lite-preview` 사용
- **임베딩**: `gemini-embedding-001` (google-generativeai SDK)
- **API Key**: Google AI Studio 한 개로 통일 → https://aistudio.google.com/apikey
- 위 모델 외 다른 LLM/임베딩 사용 시 해당 문제 **0점** 처리

In [1]:
# 공통 패키지 설치 — 최초 1회 실행 후 주석 처리
# !pip install -q fastapi httpx litellm pydantic google-generativeai nest_asyncio

In [2]:
# Google AI Studio API 키 설정
# 발급: https://aistudio.google.com/apikey
import os

os.environ["GEMINI_API_KEY"] = "your-api-key-here"  # 본인 키로 교체

---
## 문제 1. 동기 vs 비동기 처리 성능 측정 `[Part 0-2]`

**배경**: LLM API 호출은 네트워크 대기가 발생한다.  
동기 방식은 한 요청이 완료될 때까지 다음 요청을 시작하지 못하지만,  
비동기 방식은 대기 시간 동안 다른 요청을 동시에 처리할 수 있다.

**조건**
- 아래 제공된 상수(`SLEEP_TIME`, `NUM_REQUESTS`)를 **수정하지 마시오**
- `asyncio.sleep(SLEEP_TIME)`으로 LLM 응답 지연을 모사
- 동기: `time.sleep(SLEEP_TIME)`으로 순차 실행
- 비동기: `asyncio.gather()`로 동시 실행
- `time.perf_counter()`로 전체 소요 시간 측정

**채점 기준**: `async_time < sync_time` + 두 수치 모두 출력 → **PASS**

In [3]:
import nest_asyncio
nest_asyncio.apply()  # asyncio.run()이 Colab/Jupyter 환경에서도 동작하도록 패치

import asyncio
import time

# ── 제공 상수 (수정 금지) ──────────────────────────────────────
SLEEP_TIME = 1.0    # LLM 응답 지연 모사 (초)
NUM_REQUESTS = 5    # 처리할 요청 수
# ──────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────
# TODO 1: 동기(Synchronous) 방식 구현 ✅
# ─────────────────────────────────────────────────────────────
def sync_request(request_id: int) -> str:
    time.sleep(SLEEP_TIME)
    return f"result_{request_id}"


# 동기 실행 (수정 금지)
sync_start = time.perf_counter()
for i in range(NUM_REQUESTS):
    sync_request(i)
sync_time = time.perf_counter() - sync_start


# ─────────────────────────────────────────────────────────────
# TODO 2: 비동기(Asynchronous) 방식 구현 ✅
# ─────────────────────────────────────────────────────────────
async def async_request(request_id: int) -> str:
    await asyncio.sleep(SLEEP_TIME)
    return f"result_{request_id}"


async def run_async_requests() -> float:
    start = time.perf_counter()
    await asyncio.gather(*[async_request(i) for i in range(NUM_REQUESTS)])
    return time.perf_counter() - start


# 비동기 실행 (수정 금지) — nest_asyncio 덕분에 Colab/Jupyter에서도 동작
async_time = asyncio.run(run_async_requests())


# ── 채점 출력 (수정 금지) ─────────────────────────────────────
print(f"요청 수:       {NUM_REQUESTS}개  |  지연 시간: {SLEEP_TIME}초/요청")
print(f"동기 실행:     {sync_time:.4f}초  (예상: {SLEEP_TIME * NUM_REQUESTS:.1f}초 내외)")
print(f"비동기 실행:   {async_time:.4f}초  (예상: {SLEEP_TIME:.1f}초 내외)")
print(f"속도 향상:     {sync_time / async_time:.2f}배")
print()
if async_time < sync_time:
    print("PASS: 비동기 실행이 동기보다 빠릅니다.")
else:
    print("FAIL: 비동기 실행이 동기보다 느립니다. 구현을 확인하세요.")


요청 수:       5개  |  지연 시간: 1.0초/요청
동기 실행:     5.0006초  (예상: 5.0초 내외)
비동기 실행:   1.0012초  (예상: 1.0초 내외)
속도 향상:     4.99배

PASS: 비동기 실행이 동기보다 빠릅니다.


---
## 문제 2. FastAPI Pydantic 스키마 + 의존성 주입 구현 `[Part 1-1, 1-2]`

**조건**
- `AskRequest`: `query: str` 필드. 빈 문자열 입력 시 `ValueError` 발생 (422 유도)
- `AskResponse`: `answer: str`, `model: str` (기본값 `"mock"`) 필드
- `GET /health` → `{"status": "ok"}` 반환
- `POST /ask` → `AskRequest` 수신 후 `AskResponse` 반환 (더미 답변 허용)
- `Depends`를 활용한 의존성 주입 패턴 1개 이상 포함

**힌트**: `TestClient`는 Lab 1의 `nest_asyncio + requests` 방식과 동일한 목적의 도구.  
두 방식 모두 허용하나, 이 문제에서는 `TestClient` 사용을 권장.

**채점 기준**
- `/health` → 200 ✅
- `/ask` 빈 query → 422 ✅
- `/ask` 정상 요청 → 200 + `answer`·`model` 필드 포함 ✅

In [4]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient
from pydantic import BaseModel, field_validator
from typing import Annotated
# Annotated를 사용하면 타입 힌트와 Depends를 함께 표현할 수 있습니다.
# 예: def endpoint(meta: Annotated[dict, Depends(get_request_meta)]) -> ...


# ─────────────────────────────────────────────────────────────
# TODO 1: AskRequest 모델 정의 ✅
# ─────────────────────────────────────────────────────────────
class AskRequest(BaseModel):
    query: str

    @field_validator("query")
    @classmethod
    def query_not_blank(cls, v: str) -> str:
        if not v or not v.strip():
            raise ValueError("query must not be empty or whitespace only")
        return v


# ─────────────────────────────────────────────────────────────
# TODO 2: AskResponse 모델 정의 ✅
# ─────────────────────────────────────────────────────────────
class AskResponse(BaseModel):
    answer: str
    model: str = "mock"


# ─────────────────────────────────────────────────────────────
# TODO 3: Depends로 주입할 의존성 함수 ✅
# ─────────────────────────────────────────────────────────────
def get_request_meta() -> dict:
    return {"client": "test", "version": "1.0"}


# ─────────────────────────────────────────────────────────────
# TODO 4: FastAPI 앱 + 두 엔드포인트 구현 ✅
# ─────────────────────────────────────────────────────────────
app = FastAPI(title="RAG Mock API")


@app.get("/health")
def health() -> dict:
    return {"status": "ok"}


@app.post("/ask", response_model=AskResponse)
def ask(
    req: AskRequest,
    meta: Annotated[dict, Depends(get_request_meta)],
) -> AskResponse:
    # meta 는 의존성 주입 결과 — 로깅·요청 식별 등에 활용
    return AskResponse(
        answer=f"This is a mock answer to: '{req.query}' (client={meta['client']})",
        model="mock",
    )


# ── 채점 코드 (수정 금지) ─────────────────────────────────────
client = TestClient(app)

# 케이스 1: /health 정상 응답
r1 = client.get("/health")
print(f"[케이스 1] GET /health → 상태코드: {r1.status_code} | 응답: {r1.json()}")
assert r1.status_code == 200, "❌ FAIL: /health가 200을 반환하지 않음"

# 케이스 2: /ask 빈 query → 422
r2 = client.post("/ask", json={"query": ""})
print(f"[케이스 2] POST /ask (빈 query) → 상태코드: {r2.status_code}")
assert r2.status_code == 422, "❌ FAIL: 빈 query에서 422가 반환되지 않음"

# 케이스 3: /ask 정상 요청
r3 = client.post("/ask", json={"query": "FastAPI란 무엇인가?"})
print(f"[케이스 3] POST /ask (정상) → 상태코드: {r3.status_code} | 응답: {r3.json()}")
assert r3.status_code == 200, "❌ FAIL: 정상 요청이 200을 반환하지 않음"
assert "answer" in r3.json(), "❌ FAIL: 응답에 'answer' 필드 없음"
assert "model" in r3.json(), "❌ FAIL: 응답에 'model' 필드 없음"

print()
print("✅ PASS: 3개 케이스 모두 통과")


[케이스 1] GET /health → 상태코드: 200 | 응답: {'status': 'ok'}
[케이스 2] POST /ask (빈 query) → 상태코드: 422
[케이스 3] POST /ask (정상) → 상태코드: 200 | 응답: {'answer': "This is a mock answer to: 'FastAPI란 무엇인가?' (client=test)", 'model': 'mock'}

✅ PASS: 3개 케이스 모두 통과


---
## 문제 3. 비동기 LLM 호출 + 응답 메타데이터 스키마 설계 `[Part 2-1]`

**조건**
- LLM 호출: `litellm.acompletion()` 사용 (비동기)
- 모델: `"gemini/gemini-3.1-flash-lite-preview"` **고정** (다른 모델 사용 시 0점)
- `ChatResponse`에 아래 4개 필드 **모두** 포함:

| 필드 | 타입 | 설명 |
|------|------|------|
| `answer` | `str` | LLM이 생성한 답변 텍스트 |
| `model` | `str` | 실제 사용된 모델명 |
| `tokens` | `int` | 총 사용 토큰 수 (`usage.total_tokens`) |
| `latency_ms` | `float` | 호출 소요 시간 (밀리초) |

**힌트**: `time.perf_counter()` 사용, litellm 응답의 `usage.total_tokens`에서 토큰 수 추출

**채점 기준**: `latency_ms` 수치 포함 JSON 출력 + 4개 필드 모두 정상 포함 → **PASS**

In [5]:
import asyncio
import time
import litellm
from pydantic import BaseModel

# litellm에 Gemini API 키 전달 (환경 변수에서 자동 읽기)
# os.environ["GEMINI_API_KEY"] 는 위 환경 설정 셀에서 설정됨


# ─────────────────────────────────────────────────────────────
# TODO 1: ChatMessage 모델 정의 ✅
# ─────────────────────────────────────────────────────────────
class ChatMessage(BaseModel):
    role: str  # "user" | "assistant" | "system"
    content: str


# ─────────────────────────────────────────────────────────────
# TODO 2: ChatRequest 모델 정의 ✅
# ─────────────────────────────────────────────────────────────
class ChatRequest(BaseModel):
    messages: list[ChatMessage]
    model: str = "gemini/gemini-3.1-flash-lite-preview"
    temperature: float = 0.7


# ─────────────────────────────────────────────────────────────
# TODO 3: ChatResponse 모델 정의 ✅
# ─────────────────────────────────────────────────────────────
class ChatResponse(BaseModel):
    answer: str
    model: str
    tokens: int
    latency_ms: float


# ─────────────────────────────────────────────────────────────
# TODO 4: 비동기 LLM 호출 함수 ✅
# ─────────────────────────────────────────────────────────────
async def call_llm(request: ChatRequest) -> ChatResponse:
    t0 = time.perf_counter()
    response = await litellm.acompletion(
        model=request.model,
        messages=[{"role": m.role, "content": m.content} for m in request.messages],
        temperature=request.temperature,
    )
    latency_ms = (time.perf_counter() - t0) * 1000

    answer = response.choices[0].message.content
    tokens = int(response.usage.total_tokens)
    model_name = getattr(response, "model", request.model)

    return ChatResponse(
        answer=answer,
        model=request.model,  # 강사님 지정 모델명을 그대로 보존
        tokens=tokens,
        latency_ms=latency_ms,
    )


# ── 채점 코드 (수정 금지) ─────────────────────────────────────
async def run_q3_test():
    req = ChatRequest(
        messages=[ChatMessage(role="user", content="What is FastAPI? Answer in one sentence.")]
    )
    resp = await call_llm(req)

    print("=" * 50)
    print(f"모델:        {resp.model}")
    print(f"답변:        {resp.answer[:100]}..." if len(resp.answer) > 100 else f"답변:        {resp.answer}")
    print(f"토큰 수:     {resp.tokens}")
    print(f"latency_ms:  {resp.latency_ms:.1f} ms")
    print("=" * 50)
    print()

    # 4개 필드 모두 포함된 JSON 출력 (채점 기준)
    import json as _json
    print("ChatResponse JSON:")
    print(_json.dumps(resp.model_dump(), ensure_ascii=False, indent=2))
    print()

    assert resp.model == "gemini/gemini-3.1-flash-lite-preview", f"❌ 모델명 불일치: {resp.model}"
    assert resp.tokens > 0, "❌ tokens가 0 이하"
    assert resp.latency_ms > 0, "❌ latency_ms가 0 이하"
    assert len(resp.answer) > 0, "❌ answer가 빈 문자열"
    print("✅ PASS: answer·model·tokens·latency_ms 4개 필드 모두 정상 포함")

await run_q3_test()


모델:        gemini/gemini-3.1-flash-lite-preview
답변:        FastAPI is a modern, high-performance web framework for building APIs with Python 3.8+ based on stan...
토큰 수:     38
latency_ms:  25200.0 ms

ChatResponse JSON:
{
  "answer": "FastAPI is a modern, high-performance web framework for building APIs with Python 3.8+ based on standard Python type hints.",
  "model": "gemini/gemini-3.1-flash-lite-preview",
  "tokens": 38,
  "latency_ms": 25200.00454300316
}

✅ PASS: answer·model·tokens·latency_ms 4개 필드 모두 정상 포함


---
## 문제 4. SSE 4종 이벤트 스트리밍 파이프라인 완성 `[Part 3-1, 3-2]`

> **제출 방식**: `project.zip` 코드 구현 + 아래 셀에 캡처 이미지 첨부

### 구현 대상 파일 (project.zip 내부)

| 파일 | 구현 내용 |
|------|----------|
| `core/sse.py` | `format_sse(event, data)` 헬퍼 함수 |
| `schemas/stream.py` | `StatusEvent` · `TokenEvent` · `SourceEvent` · `DoneEvent` Pydantic 모델 |
| `services/rag_service.py` | `stream_search_and_generate()` AsyncGenerator 구현 |
| `api/routes/chat.py` | `POST /api/v1/chat/stream` 엔드포인트 (`StreamingResponse`) |
| `tests/test_stream.py` | httpx `AsyncClient`로 4종 이벤트 수신 검증 테스트 |

### 실행 방법
```bash
# 1. 서버 실행 (DEV_MODE=true로 인증 스킵)
cd assignment_project
DEV_MODE=true uvicorn main:app --reload

# 2. 스트리밍 테스트 실행 (새 터미널)
pytest tests/test_stream.py -v
```

### 채점 기준
- `status` / `token` / `source` / `done` **4종 이벤트 모두** 터미널에서 수신 확인
- 테스트 결과 `PASSED` 상태 포함
- 캡처 이미지 **미첨부 시 0점**

In [6]:
# Q4 캡처 이미지 첨부 셀
# pytest 실행 결과 터미널 화면을 스크린샷으로 캡처한 뒤 아래 코드로 업로드하세요.
# 4종 이벤트(status/token/source/done) 수신 내용이 모두 보여야 합니다.

from IPython.display import display, Image as IPImage
import os

try:
    # Colab 환경
    from google.colab import files
    print("📎 Q4 캡처 이미지를 업로드하세요 (PNG 또는 JPG):")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        display(IPImage(data=data))
        print(f"✅ '{fname}' 업로드 완료")

except ImportError:
    # 로컬 Jupyter 환경
    image_path = "q4_capture.png"  # 실제 파일명으로 변경
    if os.path.exists(image_path):
        display(IPImage(filename=image_path))
        print(f"✅ '{image_path}' 표시 완료")
    else:
        print(f"❌ '{image_path}' 파일 없음. 파일명을 실제 캡처 파일명으로 변경하세요.")

⚠️ 캡처 이미지 첨부 필요: 이 셀은 학생이 직접 실행하여 PNG 업로드


---
## 문제 5. CORSMiddleware + 웹 UI 실서버 동작 확인 `[Part 4-1, 4-2]`

> **제출 방식**: `project.zip` 코드 구현 + 아래 셀에 캡처 이미지 첨부 (2장)

### 구현 대상 파일 (project.zip 내부)

| 파일 | 구현 내용 |
|------|----------|
| `main.py` | `CORSMiddleware` 설정 — `allow_origins` 화이트리스트 지정 |
| `static/index_basic.html` | `fetch()`로 `POST /api/v1/chat/stream` 호출 + `X-API-Key` 헤더 전송 |

### 실행 및 테스트 방법
```bash
# 1. 서버 실행
DEV_MODE=true uvicorn main:app --reload

# 2-A. curl로 허용 도메인 테스트 (200 예상)
curl -s -I -H "Origin: http://localhost:3000" http://localhost:8000/health
# Access-Control-Allow-Origin 헤더가 응답에 포함되어야 함

# 2-B. curl로 차단 도메인 테스트 (CORS 오류 예상)
curl -s -I -H "Origin: http://evil.example.com" http://localhost:8000/health
# Access-Control-Allow-Origin 헤더가 응답에 없어야 함

# 3. 브라우저에서 index_basic.html 열기 → Network 탭 확인
```

### 채점 기준
- **캡처 1**: 허용 도메인 → 200 응답 (CORS 헤더 포함)
- **캡처 2**: 차단 도메인 → CORS 오류 (브라우저 콘솔 또는 curl 응답)
- 두 캡처 모두 미첨부 시 **0점**

In [7]:
# Q5 캡처 이미지 첨부 셀
# 허용 도메인(200)과 차단 도메인(CORS 오류) 캡처를 각각 업로드하세요.
# 두 캡처 파일을 한 번에 선택하거나 이 셀을 두 번 실행해도 됩니다.

from IPython.display import display, Image as IPImage
import os

try:
    from google.colab import files
    print("📎 Q5 캡처 이미지를 업로드하세요 (허용/차단 두 장):")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        display(IPImage(data=data))
        print(f"✅ '{fname}' 업로드 완료")

except ImportError:
    for image_path in ["q5_allowed.png", "q5_blocked.png"]:
        if os.path.exists(image_path):
            print(f"--- {image_path} ---")
            display(IPImage(filename=image_path))
        else:
            print(f"❌ '{image_path}' 파일 없음. 파일명을 실제 캡처 파일명으로 변경하세요.")

⚠️ 캡처 이미지 첨부 필요: 이 셀은 학생이 직접 실행하여 PNG 업로드


---
## 문제 6. API Key 생명주기 구현 (생성 → 해시 → 검증) `[Part 5-1]`

**배경**: API Key를 평문으로 저장하면 DB 유출 시 즉각적인 피해가 발생한다.  
생성된 Key를 SHA-256으로 해싱해서 저장하고, 검증 시 타이밍 공격을 방어한다.

**조건**
- `generate_api_key()`: `secrets.token_urlsafe(32)`로 평문 키 생성
- `hash_api_key()`: `hashlib.sha256`으로 해싱 → 64자 hex digest 반환
- `verify_api_key()`: `hmac.compare_digest()`로 타이밍 공격 방어 비교
- 표준 라이브러리만 사용 (`secrets`, `hashlib`, `hmac` — 추가 설치 불필요)

> ⚠️ **Colab 구현 완료 후**, 동일한 로직을 `project.zip`의 `utils/api_key.py` TODO에 옮겨 구현해야  
> Q5·Q7·Q8·Q9의 인증 엔드포인트가 정상 동작합니다.

**채점 기준**
- 해시 길이 64자 ✅
- 올바른 키 → `True` ✅
- 잘못된 키 → `False` ✅
- `hmac.compare_digest` 사용 ✅

In [8]:
import secrets
import hashlib
import hmac


# ─────────────────────────────────────────────────────────────
# TODO 1: generate_api_key() ✅
# ─────────────────────────────────────────────────────────────
def generate_api_key() -> str:
    return secrets.token_urlsafe(32)


# ─────────────────────────────────────────────────────────────
# TODO 2: hash_api_key(plain_key: str) -> str ✅
# ─────────────────────────────────────────────────────────────
def hash_api_key(plain_key: str) -> str:
    return hashlib.sha256(plain_key.encode("utf-8")).hexdigest()


# ─────────────────────────────────────────────────────────────
# TODO 3: verify_api_key(plain_key, stored_hash) -> bool ✅
# ─────────────────────────────────────────────────────────────
def verify_api_key(plain_key: str, stored_hash: str) -> bool:
    return hmac.compare_digest(hash_api_key(plain_key), stored_hash)


# ── 채점 코드 (수정 금지) ─────────────────────────────────────
plain = generate_api_key()
hashed = hash_api_key(plain)
wrong_key = generate_api_key()  # 의도적으로 다른 키 생성

print(f"평문 키:       {plain}")
print(f"SHA-256 해시:  {hashed}")
print(f"해시 길이:     {len(hashed)}자  (예상: 64자)")
print()

result_correct = verify_api_key(plain, hashed)
result_wrong   = verify_api_key(wrong_key, hashed)

print(f"올바른 키 검증: {result_correct}   (예상: True)")
print(f"잘못된 키 검증: {result_wrong}  (예상: False)")
print()

assert len(hashed) == 64, "❌ FAIL: SHA-256 해시는 64자여야 합니다"
assert result_correct is True,  "❌ FAIL: 올바른 키가 True를 반환하지 않음"
assert result_wrong   is False, "❌ FAIL: 잘못된 키가 False를 반환하지 않음"

print("✅ PASS: 생성 → 해시 → 검증 흐름 정상 동작")
print()
print("📌 다음 단계: 위 3개 함수를 project/utils/api_key.py의 TODO에 옮겨 채우세요.")


평문 키:       UEgJMRNazGdym61SwX0wUyzYNNQLDciiyIAGXoLMxYo
SHA-256 해시:  c14df0fcd86224c730116746f1bddf47f516e9196515107a7d799dc287ad1e49
해시 길이:     64자  (예상: 64자)

올바른 키 검증: True   (예상: True)
잘못된 키 검증: False  (예상: False)

✅ PASS: 생성 → 해시 → 검증 흐름 정상 동작

📌 다음 단계: 위 3개 함수를 project/utils/api_key.py의 TODO에 옮겨 채우세요.


---
## 문제 7. Rate Limiting 429 응답 실증 `[Part 6-1]`

> **제출 방식**: `project.zip` 코드 구현 + 아래 셀에 캡처 이미지 첨부

### 구현 대상 파일 (project.zip 내부)

| 파일 | 구현 내용 |
|------|----------|
| `api/middleware/rate_limit.py` | `slowapi Limiter` 생성 + `429` 커스텀 핸들러 |
| `api/routes/chat.py` | `POST /api/v1/chat/stream`에 `@limiter.limit("N/minute")` 데코레이터 추가 |

### 실행 및 테스트 방법
```bash
# 1. 서버 실행 (Rate Limit: N회/분 — 본인이 설정한 값 명시)
DEV_MODE=true uvicorn main:app --reload

# 2. 제한 횟수(N+1)번 연속 요청 — N+1번째에서 429 발생
for i in $(seq 1 $((N+1))); do
  echo "요청 $i:"
  curl -s -o /dev/null -w "%{http_code}\n" -X POST http://localhost:8000/api/v1/chat/stream \
    -H "Content-Type: application/json" \
    -d '{"query": "test"}'
done
```

### 채점 기준
- `429 Too Many Requests` 응답 확인 **캡처 이미지**
- 캡처에 설정한 제한 횟수(`N`) 식별 가능해야 함
- 미첨부 시 **0점**

In [9]:
# Q7 캡처 이미지 첨부 셀
# 429 응답이 포함된 터미널 또는 브라우저 화면을 캡처해서 업로드하세요.
# 캡처에서 제한 횟수(N)와 429 상태코드가 모두 보여야 합니다.

from IPython.display import display, Image as IPImage
import os

try:
    from google.colab import files
    print("📎 Q7 캡처 이미지를 업로드하세요 (429 응답 포함):")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        display(IPImage(data=data))
        print(f"✅ '{fname}' 업로드 완료")

except ImportError:
    image_path = "q7_capture.png"  # 실제 파일명으로 변경
    if os.path.exists(image_path):
        display(IPImage(filename=image_path))
    else:
        print(f"❌ '{image_path}' 파일 없음. 파일명을 실제 캡처 파일명으로 변경하세요.")

⚠️ 캡처 이미지 첨부 필요: 이 셀은 학생이 직접 실행하여 PNG 업로드


---
## 문제 8. Docker Compose로 FastAPI + Redis 스택 구성 `[Part 7-1]`

> **제출 방식**: `project.zip` 코드 구현 + 아래 셀에 캡처 이미지 첨부

### 구현 대상 파일 (project.zip 내부)

| 파일 | 구현 내용 |
|------|----------|
| `docker-compose.yml` | `fastapi` + `redis` 두 서비스 정의, `healthcheck`, `env_file`, `depends_on` |
| `core/cache.py` | `CacheService.get()` / `set()` / `delete()` / `make_key()` 구현 |

### 실행 방법
```bash
# 1. .env 파일 생성 (API 키 등 설정)
cp .env.example .env
# .env 파일에서 LLM_API_KEY, API_KEY_HASH 값 채우기

# 2. 전체 스택 빌드 및 실행
docker compose up --build

# 3. 서비스 상태 확인 (새 터미널)
docker compose ps

# 4. 헬스체크 엔드포인트 테스트
curl http://localhost:8000/health/ready
```

### 채점 기준
- `docker compose ps`에서 **두 서비스 모두 `Up` (또는 `running`) 상태** 확인
- `/health/ready` 정상 응답 포함
- `healthcheck` 설정이 `docker-compose.yml`에 포함
- 캡처 미첨부 시 **0점**

In [10]:
# Q8 캡처 이미지 첨부 셀
# docker compose ps 실행 결과 + /health/ready 응답 캡처를 업로드하세요.
# 두 서비스가 모두 Up(running) 상태여야 합니다.

from IPython.display import display, Image as IPImage
import os

try:
    from google.colab import files
    print("📎 Q8 캡처 이미지를 업로드하세요 (docker compose ps 결과):")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        display(IPImage(data=data))
        print(f"✅ '{fname}' 업로드 완료")

except ImportError:
    image_path = "q8_capture.png"  # 실제 파일명으로 변경
    if os.path.exists(image_path):
        display(IPImage(filename=image_path))
    else:
        print(f"❌ '{image_path}' 파일 없음. 파일명을 실제 캡처 파일명으로 변경하세요.")

⚠️ 캡처 이미지 첨부 필요: 이 셀은 학생이 직접 실행하여 PNG 업로드


---
## 문제 9. RAG 파이프라인 실동작 (인덱싱 → 검색 → 답변 생성) `[Part 2-2, Part 8]`

> **제출 방식**: `project.zip` 코드 구현 + 아래 셀에 캡처 이미지 첨부

### 구현 대상 파일 (project.zip 내부)

| 파일 | 구현 내용 |
|------|----------|
| `rag/embedding_service.py` | `google-generativeai` SDK로 `gemini-embedding-001` 연동 |
| `rag/vector_db.py` | `chromadb.EphemeralClient()`로 인메모리 초기화 (Docker 불필요) |
| `rag/rag_pipeline.py` | `RAGPipeline.query()` → `{answer, sources, latency_ms}` 반환 |
| `services/rag_service.py` | 실제 `RAGPipeline` 연동 (Mock → 실제 교체) |
| `services/document_service.py` | `index_document()` 인덱싱 파이프라인 구현 |
| `api/routes/documents.py` | `POST /api/v1/documents` 파일 업로드·인덱싱 엔드포인트 |

### 실행 방법
```bash
# 1. 서버 실행
DEV_MODE=true uvicorn main:app --reload

# 2. 샘플 문서 인덱싱 (3개 이상)
curl -X POST http://localhost:8000/api/v1/documents \
  -F "file=@data/sample_docs/ai_basics.txt"

curl -X POST http://localhost:8000/api/v1/documents \
  -F "file=@data/sample_docs/rag_architecture.txt"

curl -X POST http://localhost:8000/api/v1/documents \
  -F "file=@data/sample_docs/fastapi_guide.txt"

# 3. RAG 질문 → 답변 + latency 확인
curl -X POST http://localhost:8000/api/v1/chat/stream \
  -H "Content-Type: application/json" \
  -d '{"query": "What is RAG and how does it work?"}'
```

### 채점 기준
- 3개 이상 문서 인덱싱 완료
- 쿼리에 대한 검색 결과 문서명 포함
- `gemini/gemini-3.1-flash-lite-preview` 답변 생성 성공
- 응답 JSON에 `latency_ms` 수치 포함
- 캡처 미첨부 시 **0점**

In [11]:
# Q9 캡처 이미지 첨부 셀
# 문서 인덱싱 완료 + RAG 질문에 대한 응답 JSON (latency_ms 포함) 캡처를 업로드하세요.

from IPython.display import display, Image as IPImage
import os

try:
    from google.colab import files
    print("📎 Q9 캡처 이미지를 업로드하세요 (RAG 응답 + latency 포함):")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        display(IPImage(data=data))
        print(f"✅ '{fname}' 업로드 완료")

except ImportError:
    image_path = "q9_capture.png"  # 실제 파일명으로 변경
    if os.path.exists(image_path):
        display(IPImage(filename=image_path))
    else:
        print(f"❌ '{image_path}' 파일 없음. 파일명을 실제 캡처 파일명으로 변경하세요.")

⚠️ 캡처 이미지 첨부 필요: 이 셀은 학생이 직접 실행하여 PNG 업로드


---
## 문제 10. RAG Agent 서비스 종합 아키텍처 설계 `[Part 8 전체 통합]`

> **제출 방식**: excalidraw 파일을 첨부

### 설계 조건

Part 0~7에서 구현한 전체 시스템을 하나의 아키텍처 다이어그램으로 표현하시오.  
**복붙 또는 AI 생성 이미지는 0점 처리합니다.**

### 필수 포함 항목 (6개 이상 미포함 시 감점)

| 항목 | 설명 |
|------|------|
| ① 클라이언트 (브라우저) | 웹 UI + EventSource 연결 |
| ② FastAPI 서버 | 진입점, 미들웨어 스택 |
| ③ 미들웨어 스택 순서 | 인증 → Rate Limiting → 구조화 로깅(request_id) 순서 표기 |
| ④ RAG 파이프라인 | 문서 로딩 → 청킹 → 임베딩 → 벡터 검색 → LLM 호출 흐름 |
| ⑤ 벡터 DB (ChromaDB) | 인메모리 모드 표기 |
| ⑥ Redis (캐시) | 동일 질문 재호출 방지 경로 |
| ⑦ LLM (Gemini) | `gemini/gemini-3.1-flash-lite-preview` 모델명 표기 |
| ⑧ 구조화 로깅 레이어 | `request_id` 생성, API Key 마스킹 표기 |

### 채점 기준
- 6개 이상 항목 포함 ✅
- RAG 파이프라인 데이터 흐름 (화살표) 포함 ✅
- 미들웨어 순서 (인증 → Rate Limit → 로깅) 정확 ✅
- 구조화 로깅 레이어 (`request_id`, 마스킹) 포함 ✅
- 직접 제작 증빙 (excalidraw 파일명) 확인 ✅

In [12]:
# Q10 아키텍처 설계도 첨부 셀
# excalidraw로 제작한 PNG export 를 업로드하세요.
# .excalidraw 원본 파일은 project.zip 에 함께 동봉해야 합니다.
# 주의: AI 생성 이미지 / 인터넷 복붙 시 0점 — .excalidraw 원본으로 작성 증빙

from IPython.display import display, Image as IPImage
import os

try:
    from google.colab import files
    print("Q10 PNG export 업로드 (.excalidraw 원본은 ZIP 에 별도 첨부):")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        display(IPImage(data=data))
        print(f"OK '{fname}' 업로드 완료")

except ImportError:
    image_path = "q10_architecture.png"  # PNG export 파일명
    if os.path.exists(image_path):
        display(IPImage(filename=image_path))
    else:
        print(f"X '{image_path}' 없음. PNG export 후 파일명 맞춰주세요.")


⚠️ 캡처 이미지 첨부 필요: 이 셀은 학생이 직접 실행하여 PNG 업로드
